# Data-Centric AI Agent — Training Notebook
**OpenEnv Hackathon 2026**

Trains a Qwen2.5-3B agent to improve ML datasets using:
- **Phase 1**: SFT warmup (~30 min) — teaches command grammar
- **Phase 2**: GRPO training (~3-5 hrs) — teaches strategy via RL
- **Phase 3**: Eval + reward curves

**Runtime required:** T4 GPU. `Runtime → Change runtime type → T4 GPU`

## Step 1 — Install dependencies (~5 min)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install trl datasets transformers accelerate scikit-learn pandas numpy matplotlib --quiet
!pip install "openenv-core[core]>=0.2.1" --quiet
print('Dependencies installed')

## Step 2 — Mount Google Drive (optional but recommended)
Checkpoints save every 50 GRPO steps to Drive. If Colab disconnects, you can resume.

**Skip this cell if you don't want Drive integration.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

## Step 3 — Clone / update repo + setup

In [ ]:
import os, sys, shutil

REPO     = 'https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env.git'
WORK_DIR = '/content/data-centric-env'

# Clone or pull latest
if os.path.exists(f'{WORK_DIR}/pyproject.toml'):
    print('Repo exists, pulling latest (includes new features)...')
    !git -C {WORK_DIR} pull origin main
else:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    !git clone {REPO} {WORK_DIR}

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('CWD:', os.getcwd())

# Show latest commit so we know we have the right version
!git log --oneline -3

# Install as editable package (fixes relative imports)
!pip install -e . --quiet 2>/dev/null || echo 'pip install -e . skipped'

# Symlink output dirs to Drive if mounted
DRIVE_DIR = '/content/drive/MyDrive/data-centric-training'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for d in ['data-centric-checkpoints', 'data-centric-adapter', 'logs', 'plots']:
        local  = os.path.join(WORK_DIR, d)
        remote = os.path.join(DRIVE_DIR, d)
        os.makedirs(remote, exist_ok=True)
        if not os.path.exists(local):
            os.symlink(remote, local)
            print(f'  Drive link: {d}')
        else:
            print(f'  Exists: {d}')
    print('Checkpoints will save to Drive')
else:
    print('Drive not mounted - checkpoints save locally only')

print('Setup complete')

## Step 3.5 — Verify all features work
Runs a quick smoke test to confirm all 5 new features (query_analyst, smarter specialists,
dual classifier, feature importance, drift detection) are connected and working.

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

# Quick inline smoke test (no file needed)
from models import DataCentricAction
from server.data_centric_environment import DataCentricEnvironment

env = DataCentricEnvironment()
obs = env.reset(task='task_1_easy', seed=42)

checks = {}

# 1. query_analyst
obs = env.step(DataCentricAction(message='query_analyst'))
checks['query_analyst'] = 'DIAGNOSIS' in obs.response and 'RECOMMENDED PLAN' in obs.response

# 2. Smarter specialists
obs = env.step(DataCentricAction(message='query_cleaner'))
checks['smarter_specialists'] = any(k in obs.response for k in ['skew', 'Risk:', 'Reason:', '%'])

# 3. Drift detection (apply then check)
obs = env.step(DataCentricAction(message='apply 1'))
checks['drift_detection'] = 'Distribution drift' in obs.response or 'drift' in obs.response.lower()

# 4+5. Dual classifier + feature importance
obs = env.step(DataCentricAction(message='validate'))
checks['dual_classifier'] = 'RF Accuracy' in obs.response and 'LR Accuracy' in obs.response
checks['feature_importance'] = 'Feature importance' in obs.response

print('\n=== Feature Smoke Test ===')
all_pass = True
for name, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    if not passed: all_pass = False
    print(f'  {status}: {name}')

if all_pass:
    print('\nAll 5 features verified OK - ready to train!')
else:
    print('\nWARNING: Some features failed - check git pull ran correctly')

## Step 4 — Start environment server

In [ ]:
import subprocess, time, requests, os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

# Kill any existing server on port 8000
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(2)

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

for i in range(30):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'Server ready after {i+1}s')
            break
    except:
        time.sleep(1)
else:
    server_proc.terminate()
    out, err = server_proc.communicate()
    print('STDOUT:', out.decode()[-1000:])
    print('STDERR:', err.decode()[-1000:])
    raise RuntimeError('Server failed to start in 30s - see output above')

## Step 5 — Generate SFT warmup data

In [ ]:
import os
os.chdir('/content/data-centric-env')

# Always regenerate if we updated the code (new query_analyst strategies)
# Skip only if the file is recent (within last 10 minutes)
import time
sft_path = 'sft_data.jsonl'
regen = True
if os.path.exists(sft_path):
    age_minutes = (time.time() - os.path.getmtime(sft_path)) / 60
    count = sum(1 for _ in open(sft_path))
    print(f'sft_data.jsonl exists: {count} examples, {age_minutes:.0f} min old')
    # Regenerate if old file missing query_analyst strategies (<= 18 strategies)
    regen = count < 20000  # new version has ~21 strategies

if regen:
    print('Generating SFT data (includes query_analyst strategies)...')
    !python sft_generator.py
    print('SFT data generated')
else:
    print('SFT data is up to date - skipping')

## Step 6 — Load model (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

import torch
print(f'Model loaded: {MODEL_NAME}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Step 7 — Phase 1: SFT Warmup (~30 min)

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import run_sft_warmup
model = run_sft_warmup(model, tokenizer)
print('SFT warmup complete')

## Step 8 — Phase 2: GRPO Training (~3-5 hrs on T4)

Agent learns **when** to use each command via reinforcement learning.
Checkpoints save every 50 steps. If Colab disconnects, re-run all cells —
it auto-resumes from the latest checkpoint.

**What the agent will learn:**
- Start with `query_analyst` to get a prioritised plan
- Use the Agreement signal to detect overfitting
- Focus cleaning on high-importance features
- Undo bad applies when drift is HIGH

In [ ]:
import os, sys, glob
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import run_grpo_training

# Auto-detect checkpoint to resume from
ckpt_dir = './data-centric-checkpoints'
resume_from = None
if os.path.exists(ckpt_dir):
    checkpoints = sorted(glob.glob(f'{ckpt_dir}/checkpoint-*'))
    if checkpoints:
        resume_from = checkpoints[-1]
        print(f'Resuming from: {resume_from}')
    else:
        print('No checkpoint found - starting fresh')

model = run_grpo_training(model, tokenizer, resume_from_checkpoint=resume_from)
print('GRPO training complete')

## Step 9 — Save trained model

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import save_model
save_model(model, tokenizer)
print('Model saved to ./data-centric-adapter/')

## Step 10 — Plot reward curves

In [ ]:
import os
os.chdir('/content/data-centric-env')

if os.path.exists('logs/training.jsonl'):
    lines = sum(1 for _ in open('logs/training.jsonl'))
    print(f'Training log: {lines} episodes recorded')
    !python plot_rewards.py --log logs/training.jsonl --out plots/
    from IPython.display import Image, display
    for f in ['reward_curve.png', 'success_rate.png', 'accuracy_gain.png', 'curriculum.png']:
        path = f'plots/{f}'
        if os.path.exists(path):
            print(f'\n--- {f} ---')
            display(Image(path))
else:
    print('No training log found yet - run Step 8 first')

## Step 11 — Evaluate trained agent vs random vs heuristic

In [ ]:
import os, json
os.chdir('/content/data-centric-env')

!python eval_data_centric.py

if os.path.exists('eval_results.json'):
    with open('eval_results.json') as f:
        results = json.load(f)
    print('\n=== Eval Results ===')
    for k, v in results.items():
        print(f'  {k}: {v}')

## Step 12 — Push results to GitHub

Run this **only after training + eval are done**.

In [ ]:
import os
os.chdir('/content/data-centric-env')

from getpass import getpass
token = getpass('GitHub token (repo write access): ')
repo_url = f'https://{token}@github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env.git'

!git config user.email 'training@colab.run'
!git config user.name 'Colab Training'
!git remote set-url origin {repo_url}
!git add plots/ logs/ eval_results.json 2>/dev/null || true
!git commit -m 'Add training results: reward curves + eval'
!git push origin main
print('Results pushed to GitHub')

---
## Done!

| Output | Location |
|--------|----------|
| Trained adapter | `./data-centric-adapter/` |
| Training log | `logs/training.jsonl` |
| Reward curves | `plots/*.png` |
| Eval results | `eval_results.json` |